In [1]:
import altair as alt
import pandas as pd

from pathlib import Path
from scipy.stats import kendalltau
from itertools import combinations

In [2]:
result_dir = Path("../../clax-results/2-yandex-compression")

In [3]:
def parse_results(directory):
    dfs = []
    
    for random_state in [1, 2, 3]:
        files = directory.glob(f"{random_state}/test_*.csv")
        df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
        df["random_state"] = random_state
        dfs.append(df)
        
    return pd.concat(dfs)

def load_experiment(embedding_type, embedding_name):
    full_df = parse_results(result_dir / "full")
    full_df["compression_ratio"] = "Full"
    
    dfs = [full_df.copy()]
    for compression_ratio in [2, 5, 10, 100, 1000]:
        compressed_df = parse_results(result_dir / f"{embedding_type}/{compression_ratio}")
        compressed_df["compression_ratio"] = str(compression_ratio)
        dfs.append(compressed_df)
    
    combined_df = pd.concat(dfs, ignore_index=True)
    combined_df["embedding"] = embedding_name
    combined_df["train_time_min"] = combined_df["train_time_s"] / 60
    return combined_df

hash_df = load_experiment("hash", "Hashing Trick")
qr_df = load_experiment("qr", "Quotient-Remainder")
df = pd.concat([hash_df, qr_df])
df.head()

,model,test_loss,test_ll,test_ppl,test_cond_ppl,train_time_s,random_state,compression_ratio,embedding,train_time_min
0,DBN,0.266035,-0.266025,1.343151,1.319775,1343.837146,1,Full,Hashing Trick,22.397286
1,SDBN,0.296183,-0.296174,1.360986,1.359909,1327.249178,1,Full,Hashing Trick,22.120820
2,DCM,0.297212,-0.297202,1.364831,1.361876,703.429029,1,Full,Hashing Trick,11.723817
3,CCM,0.267055,-0.267046,1.344214,1.321197,711.854675,1,Full,Hashing Trick,11.864245
4,GCTR,0.364355,-0.364347,1.489982,1.489982,173.848136,1,Full,Hashing Trick,2.897469


In [4]:
model2color = {
    "PBM": "#3182bd",
    "UBM": "#6baed6",
    "DBN": "#31a354",
    "SDBN": "#74c476",
    "CM": "#fd8d3c",
    "CCM": "#fdae6b",
    "DCM": "#fdd0a2",
    "DCTR": "#636363",
    "RCTR": "#969696",
    "GCTR": "#bdbdbd",
}
df["color"] = df.model.map(model2color)

In [5]:
import altair as alt

# Chart configuration
chart_config = {
    "width": 55,
    "height": 120,
    "title": ""
}

def create_metric_chart(filtered_df, y_field, y_title, xtitle=True):
    return alt.Chart(filtered_df, **chart_config).mark_bar().encode(
        column=alt.Column(
            "compression_ratio:N", 
            title="Compression Ratio" if xtitle else "",
            sort=["Full", "2", "5", "10", "100", "1000"], 
            spacing=3, 
            header=alt.Header(titleOrient='bottom', labelOrient='bottom', titlePadding=0)
        ),
        x=alt.X("model", sort="y", title=None).axis(labels=False, ticks=False),
        y=alt.Y(y_field, title=y_title).scale(domain=(1.3, 1.5), clamp=True),
        color=alt.Color("model", title="Models").scale(domain=list(model2color.keys()), range=list(model2color.values())),
    ).resolve_scale(x="independent", y="shared")

def create_embedding_charts(df, embedding_type, xtitle):
    filtered_df = df[df.embedding == embedding_type]
    
    cond_ppl = create_metric_chart(filtered_df, "mean(test_cond_ppl)", "Conditional PPL", xtitle)
    ppl = create_metric_chart(filtered_df, "mean(test_ppl)", "PPL", xtitle)
    combined_chart = (ppl | cond_ppl)
    return combined_chart.properties(title=f"{embedding_type} Embeddings")

hashing_trick_charts = create_embedding_charts(df, "Hashing Trick", xtitle=False)
quotient_remainder_charts = create_embedding_charts(df, "Quotient-Remainder", xtitle=True)

chart = (hashing_trick_charts & quotient_remainder_charts).configure_concat(spacing=10).configure_legend(orient="right").configure_title(offset=0, dx=55).configure_title(offset=0, dx=55)
chart.save("2-yandex-compression-models.svg")
chart

alt.VConcatChart(...)

In [6]:
def compute_rank_correlation(df):    
    results = []
    
    for metric in ["test_ppl", "test_cond_ppl"]:
        for embedding in df["embedding"].unique():
            for random_state in df["random_state"].unique():
                embedding_data = df[(df["embedding"] == embedding) & (df["random_state"] == random_state)]
                rank_matrix = embedding_data.pivot(index="model", columns="compression_ratio", values=metric)
                rank_matrix = rank_matrix.rank(ascending=True)
                
                for ratio in sorted(rank_matrix.columns):
                    if ratio != "Full":
                        corr, _ = kendalltau(rank_matrix["Full"], rank_matrix[ratio])
                        results.append({
                            "embedding": embedding,
                            "random_state": random_state,
                            "metric": metric,
                            "compression_ratio": ratio,
                            "correlation": corr,
                        })
    
    return pd.DataFrame(results)

correlation_df = compute_rank_correlation(df)
correlation_df["metric"] = correlation_df["metric"].map({"test_ppl": "Perplexity", "test_cond_ppl": "Cond. Perplexity"})
correlation_df.head()

,embedding,random_state,metric,compression_ratio,correlation
0,Hashing Trick,1,Perplexity,10,0.955556
1,Hashing Trick,1,Perplexity,100,0.911111
2,Hashing Trick,1,Perplexity,1000,0.822222
3,Hashing Trick,1,Perplexity,2,1.000000
4,Hashing Trick,1,Perplexity,5,1.000000


In [7]:
lines = alt.Chart(correlation_df, width=175, height=150).mark_line(point=True).encode(
    x=alt.X("compression_ratio", title="Compression Ratio", sort=["2", "5", "10", "100", "1000"]).axis(labelAngle=0),
    y=alt.Y("mean(correlation)", title="Rank Corr. with Full Table"),
    color=alt.Color("metric", title="", legend=alt.Legend(orient="none", legendX=75, legendY=220, direction="horizontal")),
)

error = alt.Chart(correlation_df).mark_errorband(extent="ci").encode(
    x=alt.X("compression_ratio", title="Compression Ratio", sort=["2", "5", "10", "100", "1000"]).axis(labelAngle=0),
    y=alt.Y("correlation", title=""),
    color=alt.Color("metric", title=""),
)

metric_charts = (lines + error).facet(column=alt.Column("embedding", title="", header=alt.Header(labelPadding=0, titlePadding=0))).resolve_scale(y="independent")
metric_charts

alt.FacetChart(...)

In [8]:
bars = alt.Chart(df, width=175, height=150, title=alt.Title("Training Time (mins)", offset=12)).mark_bar().encode(
    x=alt.X("compression_ratio", title="Compression Ratio", sort=["Full", "2", "5", "10", "100", "1000"]).axis(labelAngle=0),
    xOffset=alt.XOffset("embedding", title=""),
    y=alt.Y("mean(train_time_min)", title="Training time (mins)"),
color=alt.Color("embedding", title="", legend=alt.Legend(orient="none", legendX=-25, legendY=220, direction="horizontal")).scale(scheme="blues"),
)

error = alt.Chart(df).mark_errorbar(thickness=3).encode(
    x=alt.X("compression_ratio", title="Compression Ratio", sort=["Full", "2", "5", "10", "100", "1000"]).axis(labelAngle=0),
    xOffset=alt.XOffset("embedding", title=""),
    y=alt.Y("train_time_min", title="Training time (mins)"),
)

time_chart = bars + error
time_chart

alt.LayerChart(...)

In [9]:
chart = (metric_charts | time_chart).resolve_scale(color="independent")
chart

alt.HConcatChart(...)